# Day 7 Lab 01: Spark DataFrame Basics

        **Layer:** Spark fundamentals  
        **Python reference:** `Lab_Files/labs/lab_01_spark_dataframe_basics.py`

        This notebook is a sectioned companion version of the existing Python lab. It does not replace `run_lab.py` or the scripts under `Lab_Files/labs`.

        ## Learning Checkpoints
        - Start Spark from a notebook.
- Read customers.csv with the shared explicit schema.
- Apply simple DataFrame transformations and write a preview artifact.

## 0. Notebook Setup

Run this first. It moves the kernel into `Lab_Files` and makes the shared lab helpers importable.

In [1]:
from pathlib import Path
import os
import sys
import types
import typing

# PySpark 3.4 imports typing.io, which is absent in Python 3.14+.
if "typing.io" not in sys.modules:
    typing_io = types.ModuleType("typing.io")
    typing_io.IO = typing.IO
    typing_io.TextIO = typing.TextIO
    typing_io.BinaryIO = typing.BinaryIO
    sys.modules["typing.io"] = typing_io

def find_lab_files(start: Path) -> Path:
    relative_options = [
        Path("."),
        Path("Lab_Files"),
        Path("Day_07") / "Lab_Files",
        Path("Week_02") / "Day_07" / "Lab_Files",
    ]
    for root in [start, *start.parents]:
        for relative in relative_options:
            candidate = (root / relative).resolve()
            if (candidate / "labs" / "day7_common.py").exists():
                return candidate
    raise FileNotFoundError(
        "Could not find Week_02/Day_07/Lab_Files. "
        "Start Jupyter from the repository root, Week_02/Day_07, or Week_02/Day_07/Lab_Files."
    )

HERE = Path.cwd().resolve()
LAB_FILES = find_lab_files(HERE)

os.chdir(LAB_FILES)
labs_path = str(LAB_FILES / "labs")
if labs_path not in sys.path:
    sys.path.insert(0, labs_path)

print(f"Working directory: {Path.cwd()}")
print(f"Using lab helpers from: {labs_path}")


Working directory: C:\Users\Gamer\Documents\GitHub\Medallion pipeline\Week_02\Day_07\Lab_Files
Using lab helpers from: C:\Users\Gamer\Documents\GitHub\Medallion pipeline\Week_02\Day_07\Lab_Files\labs


## 1. Import Lab Helpers

In [2]:
from pyspark.sql import functions as F

import importlib
import day7_common
day7_common = importlib.reload(day7_common)


from day7_common import OUTPUT_DIR, ensure_output_dirs, read_customers, require_source_data, spark_session, write_csv_dir

## 2. Start Spark and Validate Source Data

In [3]:
require_source_data()
ensure_output_dirs()
spark = spark_session("Day7Notebook01DataFrameBasics")

## 3. Read the Customer Dimension

In [4]:
customers = read_customers(spark)
customers.printSchema()
customers.show(10, truncate=False)

root
 |-- customer_id: integer (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- country: string (nullable = true)
 |-- region: string (nullable = true)
 |-- signup_date: string (nullable = true)
 |-- loyalty_tier: string (nullable = true)

+-----------+-------------+-----------------+-------+------+-----------+------------+
|customer_id|customer_name|email            |country|region|signup_date|loyalty_tier|
+-----------+-------------+-----------------+-------+------+-----------+------------+
|501        |Ada Retail   |ada@example.com  |US     |NA    |2026-01-15 |Gold        |
|502        |Ravi Stores  |RAVI@EXAMPLE.COM |IN     |APAC  |2026-02-20 |Silver      |
|503        |Sofia Market |sofia@example.com|ES     |EMEA  |2026-03-05 |Bronze      |
|504        |Mina Travel  |mina@example.com |DE     |EMEA  |2026-01-29 |Gold        |
|505        |Omar Services|omar@example.com |AE     |MEA   |2026-04-14 |Silver      |
|506        |Nia

In [5]:
customers.show()

+-----------+-------------+-----------------+-------+------+-----------+------------+
|customer_id|customer_name|            email|country|region|signup_date|loyalty_tier|
+-----------+-------------+-----------------+-------+------+-----------+------------+
|        501|   Ada Retail|  ada@example.com|     US|    NA| 2026-01-15|        Gold|
|        502|  Ravi Stores| RAVI@EXAMPLE.COM|     IN|  APAC| 2026-02-20|      Silver|
|        503| Sofia Market|sofia@example.com|     ES|  EMEA| 2026-03-05|      Bronze|
|        504|  Mina Travel| mina@example.com|     DE|  EMEA| 2026-01-29|        Gold|
|        505|Omar Services| omar@example.com|     AE|   MEA| 2026-04-14|      Silver|
|        506|Nia Wholesale|  nia@example.com|     US|    NA| 2026-05-01|      Bronze|
+-----------+-------------+-----------------+-------+------+-----------+------------+



## 4. Transform for a Learner-Friendly Preview

In [6]:
preview = (
    customers.withColumn("email", F.lower(F.trim("email")))
    .withColumn("signup_date", F.to_date("signup_date"))
    .select("customer_id", "customer_name", "email", "region", "loyalty_tier", "signup_date")
    .orderBy("customer_id")
)

## 5. Inspect and Write Output

In [7]:
preview.show(20, truncate=False)
write_csv_dir(preview, OUTPUT_DIR / "notebook_01_customers_preview.csv")

+-----------+-------------+-----------------+------+------------+-----------+
|customer_id|customer_name|email            |region|loyalty_tier|signup_date|
+-----------+-------------+-----------------+------+------------+-----------+
|501        |Ada Retail   |ada@example.com  |NA    |Gold        |2026-01-15 |
|502        |Ravi Stores  |ravi@example.com |APAC  |Silver      |2026-02-20 |
|503        |Sofia Market |sofia@example.com|EMEA  |Bronze      |2026-03-05 |
|504        |Mina Travel  |mina@example.com |EMEA  |Gold        |2026-01-29 |
|505        |Omar Services|omar@example.com |MEA   |Silver      |2026-04-14 |
|506        |Nia Wholesale|nia@example.com  |NA    |Bronze      |2026-05-01 |
+-----------+-------------+-----------------+------+------------+-----------+



## 6. Checkpoint

In [8]:
print(f"Rows read from customers.csv: {customers.count()}")
print("Concepts: SparkSession, explicit schema, lazy transformations, action via count().")

Rows read from customers.csv: 6
Concepts: SparkSession, explicit schema, lazy transformations, action via count().


## Clean Shutdown

Stop the SparkSession when you are done with the notebook. The next notebook will create its own session.

In [9]:
# Run this at the end of the notebook, or before restarting/rerunning the lab.
spark.stop()